In [ ]:
#txtai pra semantic search e explain, shap pra explicar pipelines de classificacao
!pip install git+https://github.com/neuml/txtai#egg=txtai[pipeline] shap

In [ ]:
from txtai.embeddings import Embeddings

#busca semantica acha resultado sem nenhuma palavra em comum com a query, dificil saber o motivo do match
data = ["US tops 5 million confirmed virus cases",
        "Canada's last fully intact ice shelf has suddenly collapsed, forming a Manhattan-sized iceberg",
        "Beijing mobilises invasion craft along coast as Taiwan tensions escalate",
        "The National Park Service warns against sacrificing slower friends in a bear attack",
        "Maine man wins $1M from $25 lottery ticket",
        "Make huge profits without work, earn up to $100,000 a day"]

embeddings = Embeddings({"path": "sentence-transformers/nli-mpnet-base-v2", "content": True})

embeddings.index([(uid, text, None) for uid, text in enumerate(data)])

#explain roda a busca normal e ainda testa a importancia de cada token do texto retornado
embeddings.explain("feel good story", limit=1)

In [ ]:
from IPython.display import HTML

#explain() calcula o score removendo 1 token por vez e compara com o score original
def plot(query):
  result = embeddings.explain(query, limit=1)[0]

  output = f"<b>{query}</b><br/>"
  spans = []
  for token, score in result["tokens"]:
    color = None
    if score >= 0.1:
      color = "#fdd835"
    elif score >= 0.075:
      color = "#ffeb3b"
    elif score >= 0.05:
      color = "#ffee58"
    elif score >= 0.02:
      color = "#fff59d"

    spans.append((token, score, color))

  #garante que sempre destaca pelo menos o token mais forte, mesmo se nenhum passar dos limiares
  if result["score"] >= 0.05 and not [color for _, _, color in spans if color]:
    mscore = max([score for _, score, _ in spans])
    spans = [(token, score, "#fff59d" if score == mscore else color) for token, score, color in spans]

  for token, _, color in spans:
    if color:
      output += f"<span style='background-color: {color}'>{token}</span> "
    else:
      output += f"{token} "

  return output

HTML(plot("feel good story"))

In [ ]:
#repete o explain pra varias queries, cada uma testando um conceito diferente
output = ""
for query in ["feel good story", "climate change", "public health story", "war", "wildlife", "asia", "lucky", "dishonest junk"]:
  output += plot(query) + "<br/><br/>"

HTML(output)

In [ ]:
#roda tudo num batch so, mais eficiente que chamar explain() query por query
queries = ["feel good story", "climate change", "public health story", "war", "wildlife", "asia", "lucky", "dishonest junk"]
results = embeddings.batchexplain(queries, limit=1)

for x, result in enumerate(results):
  print(result)

In [ ]:
from txtai.app import Application

app = Application("""
writable: true
embeddings:
  path: sentence-transformers/nli-mpnet-base-v2
  content: true
""")

app.add([{"id": uid, "text": text} for uid, text in enumerate(data)])
app.index()

app.explain("feel good story", limit=1)

In [ ]:
import shap

from txtai.pipeline import Labels

#pipelines do txtai sao um wrapper em cima do hugging face, entao da pra explicar com shap direto
data = ["Dodgers lose again, give up 3 HRs in a loss to the Giants",
        "Massive dunk!!! they are now up by 15 with 2 minutes to go"]

labels = Labels(dynamic=False)

# shap testa combinacoes de tokens mascarados, nao so 1 de cada vez 
explainer = shap.Explainer(labels.pipeline) 
shap_values = explainer(data)

In [ ]:
#o quanto cada token empurra a classificacao pra negative nessa frase 
shap.plots.text(shap_values[0, :, "NEGATIVE"])

In [ ]:
#aqui a frase e positiva, entao os tokens devem contribuir bem menos pra negative
shap.plots.text(shap_values[1, :, "NEGATIVE"])